# GRUEncoder Training — SOLUSDT Phase A

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sandeep999-cyber/solusdt-ml-trader/blob/main/colab_train.ipynb)

### Workflow (no uploads needed)
1. **Open** via the badge above — Colab loads the latest version directly from GitHub
2. **Code changes** → `git push`; re-open from the badge to get the updated notebook
3. **Data** (upload once to Drive): `MyDrive/ModelProject/data_processed/`
4. **Checkpoints** auto-saved to Drive; sync locally via `scripts/pull_checkpoint.py`

In [ ]:
# Cell 0: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted at /content/drive')

In [ ]:
# Cell 1: Download code + link data
import os, urllib.request, tarfile, shutil

REPO = 'sandeep999-cyber/solusdt-ml-trader'
PROJECT_DIR = '/content/ModelProject'
ARCHIVE_URL = f'https://github.com/{REPO}/archive/main.tar.gz'

# --- Download latest code ---
if os.path.exists(PROJECT_DIR):
    runs_dir = os.path.join(PROJECT_DIR, 'model', 'runs')
    if os.path.exists(runs_dir):
        entries = [e for e in os.listdir(runs_dir) if e != '__pycache__']
        if entries:
            print(f'WARNING: model/runs/ has {len(entries)} previous run(s).')
            print('Re-running will delete them. Continue only if checkpoints'
                  ' were already saved to Drive (Cell 7).')
            print('To restart fresh: Runtime > Restart runtime, then re-run.')
    shutil.rmtree(PROJECT_DIR)

archive_path = '/tmp/repo.tar.gz'
print(f'Downloading {ARCHIVE_URL}...')
urllib.request.urlretrieve(ARCHIVE_URL, archive_path)
print('Extracting...')
with tarfile.open(archive_path) as tar:
    tar.extractall('/content')

extracted = f'/content/{REPO.split("/")[1]}-main'
os.rename(extracted, PROJECT_DIR)
os.chdir(PROJECT_DIR)
print(f'Code ready at {PROJECT_DIR}')

# --- Persist notebook to Drive ---
DRIVE_NB = '/content/drive/MyDrive/ModelProject/colab_train.ipynb'
os.makedirs(os.path.dirname(DRIVE_NB), exist_ok=True)
shutil.copy(os.path.join(PROJECT_DIR, 'colab_train.ipynb'), DRIVE_NB)
print(f'Notebook synced to Drive: {DRIVE_NB}')

# --- Link data from Drive ---
DRIVE_DATA = '/content/drive/MyDrive/ModelProject/data_processed'
LOCAL_DATA = os.path.join(PROJECT_DIR, 'data', 'processed', 'v1', 'SOLUSDT', '1m')

if not os.path.islink(LOCAL_DATA) and not (
    os.path.isdir(LOCAL_DATA) and os.listdir(LOCAL_DATA)
):
    if not os.path.isdir(DRIVE_DATA):
        raise FileNotFoundError(
            f'Data not found at {DRIVE_DATA}. '
            'Extract data_processed.tar.gz on Drive first.'
        )
    os.makedirs(os.path.dirname(LOCAL_DATA), exist_ok=True)
    try:
        os.symlink(DRIVE_DATA, LOCAL_DATA, target_is_directory=True)
        print(f'Symlinked data: {DRIVE_DATA} -> {LOCAL_DATA}')
    except OSError:
        shutil.copytree(DRIVE_DATA, LOCAL_DATA, dirs_exist_ok=True)
        print(f'Copied data to {LOCAL_DATA}')
else:
    print(f'Data available at {LOCAL_DATA}')

print(f'Files in project root: {len(os.listdir("."))}')

In [ ]:
# Cell 2: Verify GPU
import torch
print(f'PyTorch: {torch.__version__}  |  CUDA: {torch.cuda.is_available()}')
if not torch.cuda.is_available():
    raise RuntimeError('No GPU — go to Runtime > Change runtime type > T4 GPU')
print(f'GPU: {torch.cuda.get_device_name(0)}  ({torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB)')

In [ ]:
# Cell 3: Install dependencies (torch is pre-installed on Colab)
!pip install --quiet pandas pyarrow pyyaml
print('Ready.')

In [ ]:
# Cell 4: Smoke test (~10s)
smoke_ok = False
try:
    !python -m model.train --config configs/phase_a_gru_h32.yaml --smoke-test-only
    smoke_ok = True
    print('\nSmoke test PASSED.')
except Exception as e:
    print(f'Smoke test FAILED: {e}')

In [ ]:
# Cell 5: Full training — GRU h32, 30 epochs (~5 min on T4)
if smoke_ok:
    !python -m model.train --config configs/phase_a_gru_h32.yaml
else:
    print('Skipped — smoke test failed.')

---
### Diagnostic: fixed-variance GRU

Forces `log_var=0` so loss = plain MSE. If MSE drops below ~1.016
(the unconditional variance every NLL run plateaus at), the flat
mean is a variance-shortcut pathology, not a genuine ceiling.

Run cells 6a→6b after Cell 5.

In [ ]:
# Cell 6a: Diagnostic smoke test
diag_smoke_ok = False
try:
    !python -m model.train --config configs/diag_fixed_var.yaml --smoke-test-only
    diag_smoke_ok = True
    print('\nDiagnostic smoke test PASSED.')
except Exception as e:
    print(f'Diagnostic smoke test FAILED: {e}')

In [ ]:
# Cell 6b: Diagnostic full run — 15 epochs (~3 min)
if diag_smoke_ok:
    !python -m model.train --config configs/diag_fixed_var.yaml
else:
    print('Skipped — diagnostic smoke test failed.')

In [ ]:
# Cell 7: Save all runs to Drive
import json, shutil
from datetime import datetime, timezone
from pathlib import Path

RUNS_DIR = Path('model/runs')
if not RUNS_DIR.exists():
    print('model/runs/ not found — no training was run.')
    raise RuntimeError('Nothing to save.')

all_dirs = sorted([d for d in RUNS_DIR.iterdir() if d.is_dir()])
real_runs = [d for d in all_dirs if 'smoketest' not in d.name]
if not real_runs:
    real_runs = all_dirs

DRIVE_RUN = Path('/content/drive/MyDrive/ModelProject/checkpoints')
DRIVE_RUN.mkdir(parents=True, exist_ok=True)

for run_dir in real_runs:
    name = run_dir.name
    best = run_dir / 'checkpoints' / 'best.pt'
    if not best.exists():
        print(f'  {name}: no best.pt')
        continue

    shutil.copy(best, DRIVE_RUN / f'{name}_best.pt')
    print(f'  {name}/best.pt -> Drive')

    config = run_dir / 'config.yaml'
    if config.exists():
        shutil.copy(config, DRIVE_RUN / f'{name}_config.yaml')

    metrics = run_dir / 'metrics.jsonl'
    if metrics.exists():
        shutil.copy(metrics, DRIVE_RUN / f'{name}_metrics.jsonl')
        with open(metrics) as f:
            lines = [l for l in f if l.strip()]
        if lines:
            last = json.loads(lines[-1])
            print(f'    epoch={last.get("epoch","?")}  mse={last.get("mse","?")}  loss={last.get("loss","?")}')

# Update latest pointer for local sync
latest = real_runs[-1].name
latest_best = RUNS_DIR / latest / 'checkpoints' / 'best.pt'
if latest_best.exists():
    shutil.copy(latest_best, DRIVE_RUN / 'best.pt')
    pointer = {
        'run_name': latest,
        'checkpoint_path': str(DRIVE_RUN / f'{latest}_best.pt'),
        'updated': datetime.now(timezone.utc).isoformat(),
    }
    with open(DRIVE_RUN / 'latest.json', 'w') as f:
        json.dump(pointer, f, indent=2)
    print(f'\nPointer -> {latest}')

print(f'\nResults saved. Open locally: python scripts/watch_checkpoints.py')

In [ ]:
# Cell 8: Download checkpoint (optional manual download)
from pathlib import Path
from google.colab import files

RUNS_DIR = Path('model/runs')
dirs = sorted([d for d in RUNS_DIR.iterdir() if d.is_dir() and 'smoketest' not in d.name])
if not dirs:
    dirs = sorted([d for d in RUNS_DIR.iterdir() if d.is_dir()])
latest = dirs[-1] if dirs else None

if latest:
    best_pt = latest / 'checkpoints' / 'best.pt'
    if best_pt.exists():
        files.download(str(best_pt))
        print(f'Downloading {latest.name}/best.pt...')
    else:
        print(f'No best.pt in {latest.name}')
else:
    print('No runs found')